# Qwen3.5-9B 640px Full VQA — Fresh Training

이 노트북은 기존 512 adapter를 이어서 학습하지 않습니다. `Qwen/Qwen3.5-9B` base에
새 LoRA를 붙이고 640px 입력으로 처음부터 3epoch 학습합니다.

- A100 80GB 전용
- 640px vision+language LoRA 신규 학습
- `[16, 12, 8, 4]` 중 안전하게 들어가는 가장 큰 micro-batch 자동 선택
- zero-shot 추론 생략
- 매 epoch 전체 validation 평가 및 best checkpoint 자동 복원
- epoch 2와 best checkpoint의 test 확률 보존
- 640 단독 및 epoch2/best 확률 앙상블 제출 생성
- 모든 중요 결과와 adapter를 Google Drive에 백업

새 Colab 런타임에서 GPU를 A100 80GB로 설정한 뒤 **런타임 → 모두 실행**하세요.


## 1. 충돌 없는 환경 설치


In [ ]:
import sys, subprocess, importlib
import importlib.metadata as metadata

TARGET_VERSIONS = {"transformers": "5.15.1", "peft": "0.20.0", "Pillow": "11.3.0"}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

torchao_version = installed_version("torchao")
if torchao_version is not None:
    assert "torchao" not in sys.modules, (
        f"torchao {torchao_version}가 이미 로드되었습니다. 런타임을 삭제하고 다시 실행하세요."
    )
    print(f"사용하지 않는 충돌 패키지 torchao {torchao_version} 제거 중...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    importlib.invalidate_caches()
assert installed_version("torchao") is None

before = {package: installed_version(package) for package in TARGET_VERSIONS}
ready = all(before[package] == version for package, version in TARGET_VERSIONS.items())
if not ready:
    preloaded = [name for name in ("transformers", "peft", "PIL") if name in sys.modules]
    if preloaded:
        raise RuntimeError(
            f"설치 전 이미 import된 패키지가 있습니다: {preloaded}. "
            "런타임 > 연결 해제 및 런타임 삭제 후 다시 모두 실행하세요."
        )
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade",
        "transformers==5.15.1", "peft==0.20.0", "accelerate>=1.14.0",
        "safetensors>=0.6.0", "Pillow==11.3.0", "pandas>=2.2", "tqdm>=4.66",
    ], check=True)

after = {package: installed_version(package) for package in TARGET_VERSIONS}
assert after == TARGET_VERSIONS, f"패키지 버전 불일치: {after}"
print("환경 준비 완료:", after)


## 2. 640px 학습 설정


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, math, random, shutil, zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
Image.MAX_IMAGE_PIXELS = None

assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."
DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert GPU_VRAM_GIB >= 70, f"A100 80GB가 필요합니다: {GPU_NAME}, {GPU_VRAM_GIB:.1f} GiB"
print("GPU:", GPU_NAME, f"{GPU_VRAM_GIB:.1f} GiB")

MODEL_ID = "Qwen/Qwen3.5-9B"
DATA_ROOT = Path("/content")
DATA_ARCHIVE = Path("/content/drive/MyDrive/2026-ssafy-15-2-ai.zip")
IMAGE_SIZE = 640
EPOCHS = 3

# A100 80GB에서 가장 큰 안전 batch를 smoke test로 자동 선택합니다.
TRAIN_BATCH_CANDIDATES = [16, 12, 8, 4]
MEMORY_SAFETY_RATIO = 0.88
INFER_BATCH_SIZE = 8
NUM_WORKERS = 0

BASE_BATCH_SIZE = 8
BASE_LEARNING_RATE = 5e-5
MAX_LEARNING_RATE = 7e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MIN_DEV_VOTES = 4
PSEUDO_CAP_PER_CATEGORY = 0.75
AUGMENT_HORIZONTAL_FLIP = True
RUN_EPOCH2_TEST = True
EXPORT_TO_DRIVE = True

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("/content/qwen35_9b_640") / RUN_STAMP
BEST_ADAPTER_DIR = OUTPUT_ROOT / "best_adapter"
HISTORY_PATH = OUTPUT_ROOT / "training_history.csv"
BEST_VALID_PATH = OUTPUT_ROOT / "qwen35_640_best_valid.csv"
BEST_TEST_PATH = OUTPUT_ROOT / "qwen35_640_best_test.csv"
BEST_SUBMISSION_PATH = OUTPUT_ROOT / "submission_qwen35_640_best.csv"
EPOCH2_TEST_PATH = OUTPUT_ROOT / "qwen35_640_epoch2_test.csv"
EPOCH2_SUBMISSION_PATH = OUTPUT_ROOT / "submission_qwen35_640_epoch2.csv"
BLEND_SUBMISSION_PATH = OUTPUT_ROOT / "submission_qwen35_640_epoch2_best_blend.csv"
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/qwen35_9b_640")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LETTERS = ["a", "b", "c", "d"]
LETTER_TO_INDEX = {letter: index for index, letter in enumerate(LETTERS)}
PROB_COLUMNS = [f"prob_{letter}" for letter in LETTERS]

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()

def cuda_memory():
    return {
        "allocated_GB": round(torch.cuda.memory_allocated() / 1e9, 2),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1e9, 2),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

print("fresh 640 training / output:", OUTPUT_ROOT)


## 3. 데이터 준비와 고정 validation


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

required = [
    DATA_ROOT / "train.csv", DATA_ROOT / "dev.csv", DATA_ROOT / "test.csv",
    DATA_ROOT / "train", DATA_ROOT / "dev", DATA_ROOT / "test",
]
if not all(path.exists() for path in required):
    assert DATA_ARCHIVE.exists(), f"데이터 ZIP이 없습니다: {DATA_ARCHIVE}"
    print("데이터 압축 해제:", DATA_ARCHIVE)
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall(DATA_ROOT)
missing = [str(path) for path in required if not path.exists()]
assert not missing, f"필수 데이터 누락: {missing}"

train_df = pd.read_csv(DATA_ROOT / "train.csv")
dev_df = pd.read_csv(DATA_ROOT / "dev.csv")
test_df = pd.read_csv(DATA_ROOT / "test.csv")

def categorize(question):
    question = str(question)
    if "몇 개" in question or "개수" in question:
        return "counting"
    if "재질" in question or "소재" in question:
        return "material"
    if "색" in question:
        return "color"
    if "종류" in question:
        return "type"
    return "other"

for frame in (train_df, dev_df, test_df):
    frame["category"] = frame["question"].map(categorize)

assert train_df["answer"].isin(LETTERS).all()
assert train_df["id"].is_unique and dev_df["id"].is_unique and test_df["id"].is_unique
split_index = int(len(train_df) * 0.9)
gold_train_df = train_df.iloc[:split_index].copy().reset_index(drop=True)
valid_df = train_df.iloc[split_index:].copy().reset_index(drop=True)
assert len(valid_df) == 508

def image_path(relative_path):
    path = Path(str(relative_path))
    return path if path.is_absolute() else DATA_ROOT / path

for frame in (gold_train_df, valid_df, test_df):
    missing_images = [str(x) for x in frame["path"].head(100) if not image_path(x).exists()]
    assert not missing_images, f"이미지 경로 실패: {missing_images[:3]}"

print("gold train:", len(gold_train_df), "/ valid:", len(valid_df),
      "/ dev:", len(dev_df), "/ test:", len(test_df))
print("valid category\n", valid_df["category"].value_counts())
print("test category\n", test_df["category"].value_counts())


## 4. 프롬프트와 평가 함수


In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an expert visual multiple-choice question answering system. "
    "Inspect the entire image carefully. For quantity questions, count every relevant visible object exactly once. "
    "Answer with exactly one lowercase letter: a, b, c, or d. Do not explain."
)

def build_prompt(row):
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n\n"
        "정답을 a, b, c, d 중 한 글자로만 출력하세요."
    )

def build_messages(row, image, answer=None):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_prompt(row)},
        ]},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(answer)}]})
    return messages

def apply_template(messages, add_generation_prompt):
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)

def evaluate_predictions(frame, title):
    correct = int((frame["answer"] == frame["pred"]).sum())
    print(f"\n=== {title} ===")
    print(f"전체: {correct}/{len(frame)} = {correct/len(frame):.4f}")
    print(frame.groupby("category")["correct"].agg(["mean", "sum", "count"]))
    print("pred distribution:", frame["pred"].value_counts().sort_index().to_dict())
    return correct

def prediction_nll(frame):
    matrix = frame[PROB_COLUMNS].to_numpy(dtype=np.float64)
    indices = frame["answer"].map(LETTER_TO_INDEX).to_numpy()
    gold_probability = matrix[np.arange(len(frame)), indices]
    return float(-np.log(np.clip(gold_probability, 1e-12, 1.0)).mean())


## 5. Qwen3.5-9B base와 640px processor 로드


In [ ]:
from transformers import AutoProcessor
try:
    from transformers import Qwen3_5ForConditionalGeneration as Qwen35Model
except ImportError:
    from transformers import AutoModelForMultimodalLM as Qwen35Model

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

model = Qwen35Model.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.eval()
MODEL_DEVICE = next(model.parameters()).device

dummy_messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
    {"role": "user", "content": [{"type": "text", "text": "Choose one: (a) A (b) B (c) C (d) D"}]},
]
dummy_prefix = apply_template(dummy_messages, add_generation_prompt=True)
dummy_ids = processor.tokenizer(dummy_prefix, add_special_tokens=False)["input_ids"]
LETTER_TOKEN_IDS = {}
for letter in LETTERS:
    extended = processor.tokenizer(dummy_prefix + letter, add_special_tokens=False)["input_ids"]
    assert extended[:len(dummy_ids)] == dummy_ids and len(extended) > len(dummy_ids)
    LETTER_TOKEN_IDS[letter] = extended[len(dummy_ids)]
assert len(set(LETTER_TOKEN_IDS.values())) == 4
print("letter token ids:", LETTER_TOKEN_IDS)

def forward_last_logits(active_model, inputs):
    try:
        return active_model(**inputs, use_cache=False, logits_to_keep=1)
    except TypeError:
        return active_model(**inputs, use_cache=False)

def score_letters(active_model, dataframe, desc):
    old_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    active_model.eval()
    probabilities = []
    letter_tensor = torch.tensor([LETTER_TOKEN_IDS[x] for x in LETTERS], device=MODEL_DEVICE)
    with torch.inference_mode():
        for start in tqdm(range(0, len(dataframe), INFER_BATCH_SIZE), desc=desc, unit="batch"):
            chunk = dataframe.iloc[start:start + INFER_BATCH_SIZE]
            images, texts = [], []
            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = ImageOps.exif_transpose(opened).convert("RGB")
                images.append(image)
                texts.append(apply_template(build_messages(row, image), add_generation_prompt=True))
            inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(MODEL_DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = forward_last_logits(active_model, inputs)
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            probs = torch.softmax(logits.float(), dim=-1)
            probabilities.extend(probs.cpu().tolist())
            del inputs, outputs, logits, probs, images, texts
    processor.tokenizer.padding_side = old_side
    result = dataframe.reset_index(drop=True).copy()
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = [row[index] for row in probabilities]
    result["pred"] = [LETTERS[int(np.argmax(row))] for row in probabilities]
    result["pred_conf"] = [float(max(row)) for row in probabilities]
    if "answer" in result:
        result["correct"] = result["pred"] == result["answer"]
    return result

print("base loaded:", MODEL_DEVICE, cuda_memory())


## 6. 신규 Vision + Language LoRA 부착


In [ ]:
from peft import LoraConfig, get_peft_model

LANGUAGE_LEAVES = {
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "in_proj_z", "out_proj",
}
VISION_LEAVES = {"qkv", "proj", "linear_fc1", "linear_fc2", "gate_proj", "up_proj", "down_proj"}

visual_targets, language_targets = [], []
for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    leaf = name.rsplit(".", 1)[-1]
    padded = f".{name}."
    if ".visual." in padded and leaf in VISION_LEAVES:
        visual_targets.append(name)
    elif ".language_model.layers." in padded and leaf in LANGUAGE_LEAVES:
        language_targets.append(name)

target_modules = visual_targets + language_targets
assert visual_targets and language_targets
print("vision targets:", len(visual_targets), visual_targets[:10])
print("language targets:", len(language_targets), language_targets[:10])

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", target_modules=target_modules, task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False
model.print_trainable_parameters()
MODEL_DEVICE = next(model.parameters()).device
print("fresh LoRA attached / memory:", cuda_memory())


## 7. Gold + 고신뢰 pseudo-label 학습셋


In [ ]:
VOTE_COLUMNS = ["answer1", "answer2", "answer3", "answer4", "answer5"]

def majority_vote(row):
    votes = [
        str(row[column]).strip().lower()
        for column in VOTE_COLUMNS
        if pd.notna(row[column]) and str(row[column]).strip().lower() in LETTERS
    ]
    if not votes:
        return pd.Series({"answer": None, "vote_count": 0, "vote_margin": 0})
    ordered = sorted(Counter(votes).items(), key=lambda item: (-item[1], item[0]))
    answer, count = ordered[0]
    second = ordered[1][1] if len(ordered) > 1 else 0
    return pd.Series({"answer": answer, "vote_count": count, "vote_margin": count - second})

vote_result = dev_df.apply(majority_vote, axis=1)
dev_labeled = dev_df.drop(columns=VOTE_COLUMNS).copy()
dev_labeled[["answer", "vote_count", "vote_margin"]] = vote_result
dev_labeled = dev_labeled[
    (dev_labeled["vote_count"] >= MIN_DEV_VOTES) & dev_labeled["answer"].isin(LETTERS)
].copy()
dev_labeled["source"] = "dev_pseudo"
gold_train_df["source"] = "gold"
gold_train_df["vote_count"] = 99
gold_train_df["vote_margin"] = 99

selected_pseudo = []
for category, gold_group in gold_train_df.groupby("category"):
    candidates = dev_labeled[dev_labeled["category"] == category].copy()
    cap = int(math.ceil(len(gold_group) * PSEUDO_CAP_PER_CATEGORY))
    candidates = candidates.sort_values(
        ["vote_count", "vote_margin"], ascending=False, kind="stable"
    ).head(cap)
    selected_pseudo.append(candidates)
selected_pseudo_df = pd.concat(selected_pseudo, ignore_index=True)

finetune_df = pd.concat([gold_train_df, selected_pseudo_df], ignore_index=True, sort=False)
finetune_df = finetune_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print("gold:", len(gold_train_df), "/ pseudo:", len(selected_pseudo_df), "/ total:", len(finetune_df))
print(finetune_df.groupby(["category", "source"]).size())


## 8. Dataset과 안전한 최대 batch 자동 선택


In [ ]:
class FullVQADataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(image_path(row["path"])) as opened:
            image = ImageOps.exif_transpose(opened).convert("RGB")
        horizontal_reference = any(
            term in str(row["question"]) for term in ["왼쪽", "오른쪽", "좌측", "우측"]
        )
        if AUGMENT_HORIZONTAL_FLIP and not horizontal_reference and random.random() < 0.5:
            image = ImageOps.mirror(image)
        return {"row": row, "image": image, "answer": str(row["answer"]).strip().lower()}

class LetterOnlyCollator:
    def __call__(self, items):
        images, texts, answers = [], [], []
        for item in items:
            row, image, answer = item["row"], item["image"], item["answer"]
            assert answer in LETTERS
            images.append(image)
            answers.append(answer)
            texts.append(apply_template(build_messages(row, image, answer), add_generation_prompt=False))

        old_side = processor.tokenizer.padding_side
        processor.tokenizer.padding_side = "right"
        encoded = processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.full_like(encoded["input_ids"], -100)
        for index, answer in enumerate(answers):
            length = int(encoded["attention_mask"][index].sum())
            token_id = LETTER_TOKEN_IDS[answer]
            candidates = torch.where(encoded["input_ids"][index, :length] == token_id)[0]
            candidates = candidates[candidates >= max(0, length - 32)]
            assert len(candidates) >= 1, f"assistant answer token 없음: {answer}"
            labels[index, int(candidates[-1])] = token_id
        encoded["labels"] = labels
        processor.tokenizer.padding_side = old_side
        return encoded

train_dataset = FullVQADataset(finetune_df)

def make_train_loader(batch_size, shuffle=True):
    return DataLoader(
        train_dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=LetterOnlyCollator(),
    )

TRAIN_BATCH_SIZE = None
batch_trials = []
for candidate in TRAIN_BATCH_CANDIDATES:
    trial_loader = make_train_loader(candidate, shuffle=False)
    trial_batch = next(iter(trial_loader))
    try:
        clear_cuda()
        torch.cuda.reset_peak_memory_stats()
        model.train()
        model.zero_grad(set_to_none=True)
        trial_batch = {
            key: value.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(value) else value
            for key, value in trial_batch.items()
        }
        with torch.autocast("cuda", dtype=torch.bfloat16):
            trial_output = model(**trial_batch, use_cache=False)
        trial_output.loss.backward()
        peak_gib = torch.cuda.max_memory_allocated() / 1024**3
        finite_gradients = [
            name for name, parameter in model.named_parameters()
            if parameter.requires_grad and parameter.grad is not None
            and torch.isfinite(parameter.grad).all() and float(parameter.grad.abs().max()) > 0
        ]
        assert any("visual" in name for name in finite_gradients)
        assert any("language_model" in name for name in finite_gradients)
        safe = peak_gib <= GPU_VRAM_GIB * MEMORY_SAFETY_RATIO
        batch_trials.append({"batch": candidate, "peak_GiB": round(peak_gib, 2), "safe": safe})
        print("batch trial:", batch_trials[-1])
        if safe:
            TRAIN_BATCH_SIZE = candidate
            break
    except (torch.OutOfMemoryError, RuntimeError) as error:
        if "out of memory" not in str(error).lower() and not isinstance(error, torch.OutOfMemoryError):
            raise
        batch_trials.append({"batch": candidate, "error": "OOM", "safe": False})
        print("batch trial OOM:", candidate)
    finally:
        model.zero_grad(set_to_none=True)
        for variable in ("trial_batch", "trial_output", "trial_loader"):
            if variable in globals():
                del globals()[variable]
        clear_cuda()

assert TRAIN_BATCH_SIZE is not None, f"안전한 batch를 찾지 못했습니다: {batch_trials}"
LEARNING_RATE = min(
    MAX_LEARNING_RATE,
    BASE_LEARNING_RATE * math.sqrt(TRAIN_BATCH_SIZE / BASE_BATCH_SIZE),
)
train_loader = make_train_loader(TRAIN_BATCH_SIZE, shuffle=True)
print("selected batch:", TRAIN_BATCH_SIZE, "/ lr:", LEARNING_RATE)
print("steps per epoch:", len(train_loader), "/ trials:", batch_trials)


## 9. 3epoch 신규 학습과 매 epoch validation


In [ ]:
from transformers import get_cosine_schedule_with_warmup
from peft import get_peft_model_state_dict, set_peft_model_state_dict

trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95)
)
updates_per_epoch = len(train_loader)
total_updates = updates_per_epoch * EPOCHS
warmup_updates = int(total_updates * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_updates, total_updates)

history = []
global_update = 0
best_epoch = None
best_valid_correct = -1
best_valid_nll = float("inf")
best_adapter_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    progress = tqdm(train_loader, total=len(train_loader), desc=f"Qwen3.5 640 epoch {epoch}")
    for step, batch in enumerate(progress, 1):
        batch = {
            key: value.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(value) else value
            for key, value in batch.items()
        }
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch, use_cache=False)
            loss = outputs.loss
        loss.backward()
        running_loss += float(loss.detach())
        torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_update += 1

        if step % 25 == 0:
            progress.set_postfix(
                loss=f"{running_loss/step:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}",
                mem=f"{torch.cuda.max_memory_allocated()/1e9:.1f}G",
            )
        del batch, outputs, loss

    clear_cuda()
    epoch_valid = score_letters(model, valid_df, f"Qwen3.5 640 valid epoch {epoch}")
    valid_correct = int(epoch_valid["correct"].sum())
    valid_nll = prediction_nll(epoch_valid)
    record = {
        "epoch": epoch, "train_loss": running_loss / len(train_loader),
        "updates": global_update, "valid_correct": valid_correct,
        "valid_accuracy": valid_correct / len(valid_df), "valid_nll": valid_nll,
    }
    for category, group in epoch_valid.groupby("category"):
        record[f"valid_{category}_correct"] = int(group["correct"].sum())
        record[f"valid_{category}_count"] = len(group)
    history.append(record)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    epoch_valid.to_csv(OUTPUT_ROOT / f"qwen35_640_valid_epoch{epoch}.csv", index=False)
    model.save_pretrained(OUTPUT_ROOT / f"adapter_epoch{epoch}", safe_serialization=True)

    is_best = (
        valid_correct > best_valid_correct
        or (valid_correct == best_valid_correct and valid_nll < best_valid_nll)
    )
    if is_best:
        best_epoch = epoch
        best_valid_correct = valid_correct
        best_valid_nll = valid_nll
        best_adapter_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in get_peft_model_state_dict(model).items()
        }
    print(record, "/ best:", best_epoch, best_valid_correct, "/ memory:", cuda_memory())
    clear_cuda()

assert best_adapter_state is not None
load_result = set_peft_model_state_dict(model, best_adapter_state)
model.save_pretrained(BEST_ADAPTER_DIR, safe_serialization=True)
print("best restored:", best_epoch, best_valid_correct, best_valid_nll, load_result)

del optimizer, scheduler, trainable_parameters, train_loader, train_dataset, best_adapter_state
clear_cuda()


## 10. Best 640 validation과 epoch 간 oracle


In [ ]:
best_valid = score_letters(model, valid_df, f"Qwen3.5 640 best epoch {best_epoch}")
best_valid.to_csv(BEST_VALID_PATH, index=False)
best_correct = evaluate_predictions(best_valid, f"Qwen3.5 640 best epoch {best_epoch}")
assert best_correct == best_valid_correct

epoch_valid_frames = {
    epoch: pd.read_csv(OUTPUT_ROOT / f"qwen35_640_valid_epoch{epoch}.csv")
    for epoch in range(1, EPOCHS + 1)
}
oracle_mask = np.zeros(len(valid_df), dtype=bool)
for frame in epoch_valid_frames.values():
    oracle_mask |= frame["pred"].to_numpy() == frame["answer"].to_numpy()
print("three-epoch oracle:", f"{oracle_mask.sum()}/508 = {oracle_mask.mean():.4f}")

diagnostics = []
for epoch, frame in epoch_valid_frames.items():
    row = {"epoch": epoch, "overall": int(frame["correct"].sum()), "nll": prediction_nll(frame)}
    for category, group in frame.groupby("category"):
        row[category] = int(group["correct"].sum())
    diagnostics.append(row)
diagnostic_table = pd.DataFrame(diagnostics)
diagnostic_table.to_csv(OUTPUT_ROOT / "epoch_diagnostics.csv", index=False)
print(diagnostic_table.to_string(index=False))


## 11. Best/epoch2 test 확률과 제출 생성


In [ ]:
from safetensors.torch import load_file

def write_submission(scored, path):
    submission = scored[["id", "pred"]].rename(columns={"pred": "answer"}).copy()
    assert len(submission) == len(test_df)
    assert submission["id"].tolist() == test_df["id"].tolist()
    assert submission["answer"].isin(LETTERS).all()
    submission.to_csv(path, index=False)
    print(path, submission["answer"].value_counts().sort_index().to_dict())
    return submission

best_test = score_letters(model, test_df, f"Qwen3.5 640 best epoch {best_epoch} test")
best_test.to_csv(BEST_TEST_PATH, index=False)
write_submission(best_test, BEST_SUBMISSION_PATH)

if RUN_EPOCH2_TEST:
    if best_epoch == 2:
        epoch2_test = best_test.copy()
    else:
        epoch2_file = OUTPUT_ROOT / "adapter_epoch2" / "adapter_model.safetensors"
        assert epoch2_file.exists()
        epoch2_state = load_file(str(epoch2_file), device="cpu")
        print("epoch2 load:", set_peft_model_state_dict(model, epoch2_state))
        del epoch2_state
        clear_cuda()
        epoch2_test = score_letters(model, test_df, "Qwen3.5 640 epoch2 test")

        best_file = BEST_ADAPTER_DIR / "adapter_model.safetensors"
        best_state = load_file(str(best_file), device="cpu")
        print("best restore:", set_peft_model_state_dict(model, best_state))
        del best_state
        clear_cuda()

    epoch2_test.to_csv(EPOCH2_TEST_PATH, index=False)
    write_submission(epoch2_test, EPOCH2_SUBMISSION_PATH)

    epoch2_valid = epoch_valid_frames[2]
    assert epoch2_valid["id"].tolist() == best_valid["id"].tolist()
    assert epoch2_test["id"].tolist() == best_test["id"].tolist()

    def normalized_probabilities(frame):
        values = frame[PROB_COLUMNS].to_numpy(dtype=np.float64)
        return values / values.sum(axis=1, keepdims=True)

    def log_blend(left, right, right_weight):
        scores = (
            (1.0 - right_weight) * np.log(np.clip(left, 1e-12, 1.0))
            + right_weight * np.log(np.clip(right, 1e-12, 1.0))
        )
        scores -= scores.max(axis=1, keepdims=True)
        values = np.exp(scores)
        return values / values.sum(axis=1, keepdims=True)

    p2_valid = normalized_probabilities(epoch2_valid)
    pb_valid = normalized_probabilities(best_valid)
    gold = best_valid["answer"].map(LETTER_TO_INDEX).to_numpy()
    weight_rows = []
    for best_weight in np.arange(0.0, 1.0001, 0.05):
        prediction = log_blend(p2_valid, pb_valid, best_weight).argmax(axis=1)
        weight_rows.append({
            "best_weight": round(float(best_weight), 2),
            "epoch2_weight": round(float(1.0 - best_weight), 2),
            "valid_correct": int((prediction == gold).sum()),
        })
    weight_table = pd.DataFrame(weight_rows)
    best_score = int(weight_table["valid_correct"].max())
    # 동률이면 standalone best에 가까운 조합을 선택합니다.
    selected_weight = float(
        weight_table[weight_table["valid_correct"] == best_score]
        .sort_values("best_weight", ascending=False).iloc[0]["best_weight"]
    )
    weight_table.to_csv(OUTPUT_ROOT / "epoch2_best_weight_search.csv", index=False)

    p2_test = normalized_probabilities(epoch2_test)
    pb_test = normalized_probabilities(best_test)
    blended_test_probability = log_blend(p2_test, pb_test, selected_weight)
    blended_test = best_test.copy()
    for index, letter in enumerate(LETTERS):
        blended_test[f"prob_{letter}"] = blended_test_probability[:, index]
    blended_test["pred"] = [LETTERS[index] for index in blended_test_probability.argmax(axis=1)]
    blended_test.to_csv(OUTPUT_ROOT / "qwen35_640_epoch2_best_blend_test.csv", index=False)
    write_submission(blended_test, BLEND_SUBMISSION_PATH)
    print("epoch2/best blend:", selected_weight, "best weight / valid", best_score)
else:
    selected_weight = None
    best_score = None


## 12. 메타데이터와 Drive 백업


In [ ]:
metadata_payload = {
    "model_id": MODEL_ID,
    "training_mode": "fresh_from_base",
    "image_size": IMAGE_SIZE,
    "seed": SEED,
    "epochs": EPOCHS,
    "selected_train_batch_size": TRAIN_BATCH_SIZE,
    "batch_trials": batch_trials,
    "learning_rate": LEARNING_RATE,
    "train_rows": len(finetune_df),
    "gold_rows": len(gold_train_df),
    "pseudo_rows": len(selected_pseudo_df),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "best_epoch": int(best_epoch),
    "best_valid_correct": int(best_correct),
    "best_valid_nll": float(best_valid_nll),
    "epoch2_best_blend_weight": selected_weight,
    "epoch2_best_blend_valid_correct": best_score,
    "gpu": GPU_NAME,
    "gpu_vram_gib": GPU_VRAM_GIB,
}
with open(OUTPUT_ROOT / "run_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata_payload, file, ensure_ascii=False, indent=2)

if EXPORT_TO_DRIVE:
    drive_run = DRIVE_OUTPUT_ROOT / RUN_STAMP
    drive_run.mkdir(parents=True, exist_ok=True)
    for artifact in OUTPUT_ROOT.iterdir():
        if artifact.is_file() and artifact.suffix in {".csv", ".json"}:
            shutil.copy2(artifact, drive_run / artifact.name)
    shutil.copytree(BEST_ADAPTER_DIR, drive_run / "best_adapter", dirs_exist_ok=True)
    epoch2_adapter = OUTPUT_ROOT / "adapter_epoch2"
    if epoch2_adapter.exists():
        shutil.copytree(epoch2_adapter, drive_run / "adapter_epoch2", dirs_exist_ok=True)
    print("Drive backup:", drive_run)

print("\n완료")
print("best valid:", f"{best_correct}/508", "/ best epoch:", best_epoch)
print("best valid probabilities:", BEST_VALID_PATH)
print("best test probabilities:", BEST_TEST_PATH)
print("best submission:", BEST_SUBMISSION_PATH)
print("local output:", OUTPUT_ROOT)
print("memory:", cuda_memory())


## 완료 후 가져올 파일

Drive의 `/MyDrive/qwen35_9b_640/<실행시각>/`에서 아래 파일을 내려받으세요.

1. `qwen35_640_best_valid.csv`
2. `qwen35_640_best_test.csv`
3. `training_history.csv`
4. `run_metadata.json`
5. `qwen35_640_epoch2_test.csv` (생성된 경우)

로컬 프로젝트의 `성능파일` 폴더에 넣으면 기존 512 모델 및 현재 0.93732 제출과
확률 단위로 앙상블할 수 있습니다.
